In [6]:
import torch
from torch import nn
from torch.nn import functional as F

双重卷积块

In [7]:
class DoubleConv(nn.Module):
    def __init__(self, inchannels, outchannels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(inchannels, outchannels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(outchannels),
            nn.ReLU(inplace=True),

            nn.Conv2d(outchannels, outchannels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(outchannels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

下采样模块

In [8]:
class Down(nn.Module):
    def __init__(self, inchannels, outchannels):
        super().__init__()
        self.conv = DoubleConv(inchannels, outchannels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        before_bool = self.conv(x)
        after_bool = self.pool(before_bool)

        return before_bool, after_bool

上采样模块

In [9]:
class Up(nn.Module):
    def __init__(self, inchannels, outchannels):
        super().__init__()
        self.up = nn.ConvTranspose2d(inchannels, outchannels)

        self.conv = DoubleConv(outchannels * 2, outchannels)

    def forward(self, x_decoder, x_encoder):
        x_up = self.up(x_decoder)

        diff_h = x_encoder.size(2) - x_up.size(2)
        diff_w = x_encoder.size(3) - x_up.size(3)

        x_up = F.pad(x_up, [
            diff_w // 2, diff_w - diff_w // 2, # 左右
            diff_h // 2, diff_h - diff_h // 2
        ])

        # 跳跃连接
        x = torch.cat([x_encoder, x_up], dim=1)

        return self.conv(x)

U-Net模型

In [10]:
class UNet(nn.Module):
    def __init__(self, inchannels=3, n_classes=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.inchannels = inchannels
        self.n_classes = n_classes

        self.enc1 = Down(inchannels, features[0])
        self.enc2 = Down(features[0], features[1])
        self.enc3 = Down(features[1], features[2])
        self.enc4 = Down(features[2], features[3])

        # bottleneck
        self.bottleneck = DoubleConv(features[3], features[3] * 2)

        self.dec4 = Up(features[3] * 2, features[3])
        self.dec3 = Up(features[3], features[2])
        self.dec2 = Up(features[2], features[1])
        self.dec1 = Up(features[1], features[0])

        # 输出层 1x1卷积
        self.head = nn.Conv2d(features[0], n_classes, kernel_size=1)

    def forward(self, x):
        enc1, x = self.enc1(x)
        enc2, x = self.enc2(x)
        enc3, x = self.enc3(x)
        enc4, x = self.enc4(x)

        x = self.bottleneck(x)

        x = self.dec4(x, enc4)
        x = self.dec3(x, enc3)
        x = self.dec2(x, enc2)
        x = self.dec1(x, enc1)

        return self.head(x)